# Step 2.3 & 2.4: Generating Embeddings & FAISS Baseline Search

Now that we can convert candidates into dense semantic strings, it's time to generate **Vector Embeddings**. 
We will use `sentence-transformers` (specifically `all-MiniLM-L6-v2` because it's CPU-friendly and keeps us under the 16GB RAM limit).

We'll also set up **FAISS** (Facebook AI Similarity Search) to index these embeddings so we can search them incredibly fast.

In [1]:
import sys
from pathlib import Path
import pandas as pd

sys.path.append('../src')
from data_ingestion import load_candidates
from features import stringify_candidate
from embeddings import EmbeddingModel, CandidateFAISS

# 1. Load Data & Stringify
sample_path = '../[PUB] India_runs_data_and_ai_challenge/India_runs_data_and_ai_challenge/sample_candidates.json'
candidates = load_candidates(sample_path)

candidate_strings = [stringify_candidate(c) for c in candidates]
print(f"Prepared {len(candidate_strings)} strings for embedding.")

Successfully loaded 50 candidates from sample_candidates.json
Prepared 50 strings for embedding.


### Generate Candidate Embeddings
This might take a few seconds as the model has to process the text. Because we are using MiniLM, it will run smoothly on your local CPU.

In [2]:
# 2. Initialize Model and Generate Embeddings
embedder = EmbeddingModel('all-MiniLM-L6-v2')

candidate_embeddings = embedder.generate_embeddings(candidate_strings)
print(f"Embedding Shape: {candidate_embeddings.shape}")

Loading SentenceTransformer model: all-MiniLM-L6-v2...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\pawar\OneDrive\Desktop\India_Runs_AI_Hackathon\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\pawar\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generating embeddings for 50 texts...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Embedding Shape: (50, 384)


### Index with FAISS
FAISS creates a highly optimized database for our vectors.

In [3]:
# 3. Initialize FAISS and add candidate embeddings
# all-MiniLM-L6-v2 outputs 384-dimensional vectors
faiss_index = CandidateFAISS(embedding_dim=384)
faiss_index.add_embeddings(candidate_embeddings)

Added 50 vectors to FAISS index. Total: 50


### Baseline Semantic Search
Now, let's load the **Job Description** (Senior AI Engineer), embed it, and use FAISS to find the top 5 candidates whose profiles semantically match the JD.

In [4]:
# 4. Load the Job Description
jd_path = '../[PUB] India_runs_data_and_ai_challenge/India_runs_data_and_ai_challenge/job_description.docx.txt'
with open(jd_path, 'r', encoding='utf-8') as f:
    job_description = f.read()
    
print(f"Loaded JD. Length: {len(job_description)} characters.\n")

# 5. Embed the JD
jd_embedding = embedder.generate_embeddings([job_description])[0]

# 6. Search FAISS
top_k = 5
distances, indices = faiss_index.search(jd_embedding, k=top_k)

print(f"\n--- TOP {top_k} SEMANTIC MATCHES ---")
for i, idx in enumerate(indices):
    candidate = candidates[idx]
    name = candidate.get('profile', {}).get('anonymized_name', 'Unknown')
    title = candidate.get('profile', {}).get('current_title', 'No Title')
    # Lower distance means closer semantic match in L2 space
    print(f"{i+1}. {name} | {title} | Distance: {distances[i]:.4f}")

Loaded JD. Length: 9497 characters.

Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


--- TOP 5 SEMANTIC MATCHES ---
1. Rahul Joshi | Project Manager | Distance: 0.7776
2. Anil Pandey | Accountant | Distance: 0.7953
3. Rajesh Desai | Business Analyst | Distance: 0.7981
4. Dev Nair | Mechanical Engineer | Distance: 0.8060
5. Pari Nair | Civil Engineer | Distance: 0.8111
